# KhadSan AI : เทรน YOLOv8 ตีกรอบผลไม้ 5 ชนิด บอกสด / เน่า

ผลไม้: แอปเปิล กล้วย ส้ม ทับทิม มะนาว (แต่ละชนิดมี Good = สด, Bad = เน่า รวม 10 คลาส)
dataset: [Fruits Quality](https://universe.roboflow.com/alex-day-qzfwa/fruits-quality-qtdtf) บน Roboflow (CC BY 4.0)

โค้ดเทรนอยู่ในโฟลเดอร์ `train/` ของ repo ทีม โน้ตบุ๊กนี้แค่ดึงโค้ดมารัน (โค้ดมีที่เดียว แก้ที่ `train/` ที่เดียวพอ)

**วิธีใช้**
1. push โฟลเดอร์ `train/` ขึ้น GitHub ก่อน (โน้ตบุ๊กดึงโค้ดจาก repo)
2. Runtime > Change runtime type > เลือก **T4 GPU** > Save
3. เตรียม dataset ตามข้อ 4 (Roboflow API key หรือไฟล์ zip ใน Drive)
4. Runtime > **Run all** แล้วรอ (ประมาณ 2-3 ชั่วโมง) ผลเก็บใน Google Drive โฟลเดอร์ `KhadSan_detect`
5. ถ้า Colab หลุดกลางทาง กด Run all ใหม่ การทดลองที่เสร็จแล้วจะถูกข้าม

## 1) เช็กว่าได้ GPU แล้ว

In [ ]:
!nvidia-smi

## 2) ติดตั้งไลบรารี

In [ ]:
!pip install -q ultralytics roboflow onnx onnxruntime
!yolo settings sync=False
import ultralytics
ultralytics.checks()

## 3) ดึงโค้ดเทรนจาก GitHub ของทีม

In [ ]:
import os
REPO = "https://github.com/srirat-intalai/khadsan-ai"

# ดึงครั้งแรก = clone / รันซ้ำ = อัปเดตเป็นโค้ดล่าสุด
if os.path.isdir("/content/khadsan-ai"):
    !git -C /content/khadsan-ai pull -q
else:
    !git clone -q {REPO} /content/khadsan-ai

CODE = "/content/khadsan-ai/train"
print(os.listdir(CODE))

## 4) ต่อ Google Drive แล้วโหลด dataset

**แบบที่ 1 ใช้ API key (แนะนำ):** กดไอคอน **กุญแจ** ทางซ้าย > Add new secret
- Name: `ROBOFLOW_API_KEY`
- Value: API key ของ Roboflow (เว็บ Roboflow > Settings > API Keys)
- เปิดสวิตช์ Notebook access

**ห้ามพิมพ์ API key ลงในโค้ดตรงๆ** เพราะโน้ตบุ๊กนี้อยู่บน GitHub

**แบบที่ 2 ไม่ใช้ API key:** zip โฟลเดอร์ `train/datasets/fruits_quality` ในเครื่อง ตั้งชื่อ `FruitsQuality.zip`
แล้วอัปโหลดไว้ที่ Drive โฟลเดอร์ `KhadSan_detect`

**ชุดทดสอบภายนอก (แนะนำ):** zip โฟลเดอร์ `train/datasets/external_test` ตั้งชื่อ `ExternalTest.zip`
อัปโหลดไว้ที่ Drive โฟลเดอร์ `KhadSan_detect` หลังเทรนเสร็จโค้ดจะเอาโมเดลไปสอบกับรูปชุดนี้ให้เอง

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

OUT = "/content/drive/MyDrive/KhadSan_detect"
os.makedirs(OUT, exist_ok=True)
SRC = "/content/FruitsQuality"

if os.path.exists(OUT + "/FruitsQuality.zip"):
    print("ใช้ไฟล์ zip จาก Drive")
    !unzip -q -o "{OUT}/FruitsQuality.zip" -d /content/zip_tmp
    # หาโฟลเดอร์ที่มี data.yaml (ไม่ว่าจะซ้อนกี่ชั้น)
    for root, dirs, files in os.walk("/content/zip_tmp"):
        if "data.yaml" in files:
            SRC = root
            break
else:
    from google.colab import userdata
    from roboflow import Roboflow
    rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
    project = rf.workspace("alex-day-qzfwa").project("fruits-quality-qtdtf")
    project.version(1).download("yolov8", location=SRC)
print("dataset อยู่ที่", SRC)

# ชุดทดสอบภายนอก (ถ้ามี)
EXTERNAL = ""
if os.path.exists(OUT + "/ExternalTest.zip"):
    !unzip -q -o "{OUT}/ExternalTest.zip" -d /content/external_tmp
    # หาโฟลเดอร์ที่มีโฟลเดอร์คลาส เช่น Apple_Good อยู่ข้างใน
    for root, dirs, files in os.walk("/content/external_tmp"):
        if "Apple_Good" in dirs:
            EXTERNAL = root
            break
    print("ชุดทดสอบภายนอกอยู่ที่", EXTERNAL)
else:
    print("ไม่มี ExternalTest.zip ข้ามการทดสอบภายนอก")

## 5) เตรียมข้อมูล
เก็บแค่ 5 ชนิด, จับกลุ่มรูปที่คล้ายกันไว้ชุดเดียวกัน (กัน data leakage), แบ่งใหม่ 70 / 15 / 15

In [ ]:
!python "{CODE}/prepare_data.py" --src "{SRC}" --out /content/data_5fruits
!cp /content/data_5fruits/dataset_counts.csv "{OUT}/"

## 6) เริ่มเทรน
7 การทดลอง: จากศูนย์ vs pretrained, SGD / Adam / RMSProp, LR คงที่ vs Cosine, Batch 16 vs 32, มี / ไม่มี weight decay

In [ ]:
!python "{CODE}/train_yolo.py" --data /content/data_5fruits/data.yaml --out "{OUT}" --external "{EXTERNAL}"

## 7) ดูผล

In [ ]:
from IPython.display import Image, Markdown, display

display(Markdown(open(OUT + "/results/summary.md", encoding="utf-8").read()))

for name in ["compare_scratch_vs_pretrained", "compare_optimizers", "compare_lr_fixed_vs_cosine",
             "compare_batch16_vs_32", "compare_weight_decay", "compare_all", "confusion_matrix_normalized_best"]:
    file = OUT + "/results/" + name + ".png"
    if os.path.exists(file):
        print(name)
        display(Image(file))

print(open(OUT + "/results/per_class_best.csv", encoding="utf-8").read())
print(open(OUT + "/models/class_info.json", encoding="utf-8").read())
if os.path.exists(OUT + "/results/external_summary.txt"):
    print(open(OUT + "/results/external_summary.txt", encoding="utf-8").read())

## 8) ลองให้โมเดลที่ได้ทายรูปใน test

In [ ]:
import glob
model = ultralytics.YOLO(OUT + "/models/best_detect.pt")
test_images = sorted(glob.glob("/content/data_5fruits/test/images/*.jpg"))[:4]
model.predict(test_images, conf=0.4, save=True, project="/content/demo", name="pred", exist_ok=True)
for f in sorted(glob.glob("/content/demo/pred/*.jpg")):
    display(Image(f, width=500))

## 9) เอาไปใช้ต่อ
- ดาวน์โหลด `KhadSan_detect/models/best_detect.pt` กับ `class_info.json` มาไว้ใน `web/models/` ของโปรเจกต์ เว็บใช้ได้เลย
- ดาวน์โหลด `KhadSan_detect/results` มาไว้ในโปรเจกต์ (กราฟ ตาราง ใส่รายงาน/สไลด์ และขึ้น GitHub เป็นหลักฐาน)
- ไม่ต้องดาวน์โหลด `runs/` (ไฟล์ใหญ่ เก็บไว้ใน Drive พอ)